# 01 - Ingest NEU images

Build plane, step 1 of the `msd_training` job.

1. **Extract** every `*.zip` found in the `raw` volume root or `raw/zip/` into
   `raw/images/<source_split>/<class>/<file>.jpg` (only images; XML annotations are skipped).
   Already extracted files with the same size are not rewritten, so re-runs are cheap.
2. **Auto Loader** (`binaryFile`, `availableNow`) incrementally appends new images to
   `image_bronze` with `label` (parent folder) and `source_split` (grandparent folder).
3. **Split** into `image_index` (deterministic, rebuilt each run):
   Kaggle `train` -> `train`; Kaggle `validation` -> half `val`, half `test` per class,
   chosen by a stable hash of the file name.
4. **Checks**: 6 classes, no duplicate paths, expected counts.

Parameters come from job parameters (bundle variables).

In [0]:
# Job parameters (set by the bundle job; defaults are for interactive runs against prod names)
dbutils.widgets.text("catalog", "poc_metal_surface_defect_diagnostics")
dbutils.widgets.text("schema", "metal_surface_defects")
dbutils.widgets.text("raw_volume", "raw")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
raw_volume = dbutils.widgets.get("raw_volume")

# The six NEU surface defect classes (folder names in the Kaggle archive)
CLASSES = ["crazing", "inclusion", "patches", "pitted_surface", "rolled-in_scale", "scratches"]
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

vol_root = f"/Volumes/{catalog}/{schema}/{raw_volume}"
images_root = f"{vol_root}/images"
checkpoint = f"{vol_root}/_checkpoints/image_bronze"
bronze_table = f"{catalog}.{schema}.image_bronze"
index_table = f"{catalog}.{schema}.image_index"

print(f"volume: {vol_root}\nbronze: {bronze_table}\nindex:  {index_table}")

In [0]:
import glob
import os
import shutil
import zipfile

zips = sorted(glob.glob(f"{vol_root}/*.zip") + glob.glob(f"{vol_root}/zip/*.zip"))
if not zips:
    raise FileNotFoundError(
        f"No .zip found in {vol_root}/ or {vol_root}/zip/. Upload the Kaggle NEU archive first."
    )

written, skipped, unknown_classes = 0, 0, set()
for zip_path in zips:
    with zipfile.ZipFile(zip_path) as archive:
        for info in archive.infolist():
            parts = info.filename.split("/")
            if info.is_dir() or not parts[-1].lower().endswith(IMAGE_EXTENSIONS):
                continue
            # Expected layout: .../<source_split>/images/<class>/<file>
            if "images" not in parts:
                continue
            i = len(parts) - 1 - parts[::-1].index("images")
            if i < 1 or i + 2 != len(parts) - 1:
                continue
            source_split, label, file_name = parts[i - 1].lower(), parts[i + 1], parts[-1]
            if label not in CLASSES:
                unknown_classes.add(label)
                continue
            target = f"{images_root}/{source_split}/{label}/{file_name}"
            if os.path.exists(target) and os.path.getsize(target) == info.file_size:
                skipped += 1
                continue
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with archive.open(info) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
            written += 1

print(f"zips: {[os.path.basename(z) for z in zips]}")
print(f"extracted: {written}, already present: {skipped}")
if unknown_classes:
    print(f"WARNING - skipped folders that are not NEU classes: {sorted(unknown_classes)}")

In [0]:
from pyspark.sql import functions as F

path_parts = F.split(F.col("path"), "/")

stream = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("pathGlobFilter", "*.{jpg,jpeg,png,bmp}")
    .load(images_root)
    .withColumn("file_name", F.element_at(path_parts, -1))
    .withColumn("label", F.element_at(path_parts, -2))
    .withColumn("source_split", F.element_at(path_parts, -3))
    .withColumn("ingested_at", F.current_timestamp())
)

query = (
    stream.writeStream.option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)
query.awaitTermination()

spark.sql(
    f"COMMENT ON TABLE {bronze_table} IS 'NEU images loaded by Auto Loader (binaryFile): "
    "path, modificationTime, length, content, file_name, label, source_split, ingested_at'"
)
print(f"{bronze_table}: {spark.table(bronze_table).count()} rows")

In [0]:
from pyspark.sql import Window

bronze = spark.table(bronze_table).dropDuplicates(["path"])

# Kaggle train -> train; Kaggle validation -> first half val, second half test per class,
# ordered by a stable hash of the file name (deterministic across runs).
by_label = Window.partitionBy("label", "source_split")
ranked = (
    bronze.withColumn("rn", F.row_number().over(by_label.orderBy(F.xxhash64("file_name"), "file_name")))
    .withColumn("n", F.count("*").over(by_label))
)
index_df = ranked.select(
    F.regexp_replace("path", "^dbfs:", "").alias("path"),
    "file_name",
    "label",
    "source_split",
    F.when(F.col("source_split") == "train", "train")
    .when((F.col("source_split") == "validation") & (F.col("rn") <= F.col("n") / 2), "val")
    .otherwise("test")
    .alias("split"),
    "length",
)

(index_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(index_table))
spark.sql(
    f"COMMENT ON TABLE {index_table} IS 'NEU image index with deterministic train/val/test split "
    "(Kaggle train -> train, Kaggle validation -> val/test 50/50 per class)'"
)

display(spark.table(index_table).groupBy("label").pivot("split", ["train", "val", "test"]).count().orderBy("label"))

In [0]:
idx = spark.table(index_table)

labels = sorted(r.label for r in idx.select("label").distinct().collect())
assert labels == sorted(CLASSES), f"Unexpected classes: {labels}"

total, distinct_paths = idx.count(), idx.select("path").distinct().count()
assert total == distinct_paths, f"Duplicate paths in index: {total - distinct_paths}"

bronze_total = spark.table(bronze_table).count()
bronze_distinct = spark.table(bronze_table).select("path").distinct().count()
assert bronze_total == bronze_distinct, f"Duplicate paths in bronze: {bronze_total - bronze_distinct}"

splits = {r.split: r["count"] for r in idx.groupBy("split").count().collect()}
assert all(splits.get(s, 0) > 0 for s in ("train", "val", "test")), f"Empty split: {splits}"

print(f"OK - {total} images, {len(labels)} classes, splits {splits}")